# OpenAI responses api
第一次尝试调用\
此处api_key取此用户环境变量(自行配置)\
不同模型参数请参考官方api文档

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
	base_url = "https://api.deepseek.com",
	api_key = os.environ.get("DEEPSEEK_API_KEY"),
)

response = client.responses.create(
	model = "deepseek-flash",
	input = "你的知识库更新到什么时候？", #此处是纯字符串调用的语法糖
	instructions = "这是一次测试,尝试调用api,流式输出",
	stream = True
)

# 为什么这么写,你直接print一下看看不就知道了ovo
for event in response:
	if event.type == "response.output_text.delta":
		print(event.delta,end="")


### 尝试传输图片
`input`既能传最简单的字符串，也能传一整组带角色、带多模态内容的输入项数组\
涉及多模态时使用`content`字段传入一个list\
音频，文件和图片的传输方式分多种，详细请阅读`openai_responses_input_content_types.md`,内有deepseek老师做详细说明及演示

In [ ]:
import os
import base64
from openai import OpenAI

# deepseek官方文档声明仅支持JPEG(JPG)、PNG、GIF、WebP
image_path = "./test_image.jpg"

def encode_image(path)->str:
	with open(path,"rb") as f:
		return base64.b64encode(f.read()).decode("utf-8")

client = OpenAI(
	base_url = "https://api.deepseek.com",
	api_key = os.environ.get("DEEPSEEK_API_KEY"),
)

response = client.responses.create(
	model = "deepseek-flash",
	input = [
		{
			"role":"user",
			"content":[
				{
					"type":"input_text",
					"text":"这是一次测试,请你描述一下这张图片的内容,如画风,人物表情和动作等",
				},
				{
					"type":"input_image",
					"image_url":f"data:image/jpeg;base64,{encode_image(image_path)}",
				},
			]
		}
	],
	instructions = "这是一次测试,尝试传输图片",
	stream = True
)


for event in response:
	if event.type == "response.output_text.delta":
		print(event.delta,end="")


### 工具调用
接着往下读之前，请确保自己已经了解过JSON Schema\
此处查询天气的api是免费的，可能需要科学上网\
具体调用函数deepseek老师看过官方文档后写的，也可以随便写一个函数模拟一下

In [ ]:
import os
from openai import OpenAI
from openai.types.responses import Response
from jsonschema import Validator,ValidationError
from pydantic import BaseModel
import requests
import json


client = OpenAI(
	api_key = os.environ.get("DEEPSEEK_API_KEY"),
	base_url= "https://api.deepseek.com"
)

WEATHER_TEXT = {
    0: "晴", 1: "基本晴", 2: "多云", 3: "阴", 45: "雾", 48: "雾凇",
    51: "毛毛雨", 53: "毛毛雨", 55: "毛毛雨", 61: "小雨", 63: "中雨", 65: "大雨",
    71: "小雪", 73: "中雪", 75: "大雪", 80: "阵雨", 81: "阵雨", 82: "强阵雨",
    95: "雷暴", 96: "雷暴伴冰雹", 99: "雷暴伴冰雹",
}

tools = []
input = []

weather_tool_schema = {
	"type": "function",
	"name": "get_weather",
	"description": "查询某城市的当前天气",
	"parameters": {
		"type": "object",
		"properties": { 
			"city": {
				"type": "string",
				"description": "城市名"
			}
		},
		"required": ["city"]
	}
}

tools.append(weather_tool_schema)

def get_weather(city : str)->str:
	geo = requests.get(
		"https://geocoding-api.open-meteo.com/v1/search",
		params={"name": city, "count": 1, "language": "zh"},
		timeout=10,
	).json()

	if not geo.get("results"):
		return f"查不到城市「{city}」，换个写法试试"
	loc = geo["results"][0]

	w = requests.get(
		"https://api.open-meteo.com/v1/forecast",
		params={"latitude": loc["latitude"], "longitude": loc["longitude"],
				"current": "temperature_2m,weather_code,wind_speed_10m"},
		timeout=10,
	).json()
	cur = w["current"]
	desc = WEATHER_TEXT.get(cur["weather_code"], f"天气码 {cur['weather_code']}")
	return f"{city} 当前{desc},{cur['temperature_2m']}°C,风速 {cur['wind_speed_10m']} km/h"

input.append(
	{
		"role":"user",
		"content":[
			{
				"type":"input_text",
				"text":"帮我查询北京的天气",
			},
		],
	},
)

response : Response = None
def call()->Response:
	return client.responses.create(
		model = "deepseek-flash",
		input = input,
		tools = tools,
		instructions = "这是一次工具调用的测试",
	)


response = call()
# 取第一个type为function_call的元素
tool_call = next(o for o in response.output if o.type == "function_call") 
print("模型想调：", tool_call.name, tool_call.arguments)


# 手动回填模型返回的消息
input.append(tool_call)
# 手动填入工具返回消息
args = json.loads(tool_call.arguments)
res = get_weather(**args)
input.append(
	{
		"type" : "function_call_output",
		"call_id" : tool_call.call_id,
		"output" : res,
	}
)

#二次调用
response = call()
print("二次调用模型返回",response.output_text)

### 循环工具调用
在达成结束条件之前，不断循环 调用请求 -> 执行函数 -> 回填\
结束条件：达到最大循环限制(max_step)或得出答案(不再调用工具)

In [ ]:
import os
from openai import OpenAI
from openai.types.responses import Response
from jsonschema import Validator,ValidationError
from pydantic import BaseModel
import requests
import json


client = OpenAI(
	api_key = os.environ.get("DEEPSEEK_API_KEY"),
	base_url= "https://api.deepseek.com"
)

WEATHER_TEXT = {
    0: "晴", 1: "基本晴", 2: "多云", 3: "阴", 45: "雾", 48: "雾凇",
    51: "毛毛雨", 53: "毛毛雨", 55: "毛毛雨", 61: "小雨", 63: "中雨", 65: "大雨",
    71: "小雪", 73: "中雪", 75: "大雪", 80: "阵雨", 81: "阵雨", 82: "强阵雨",
    95: "雷暴", 96: "雷暴伴冰雹", 99: "雷暴伴冰雹",
}

tools : list = []

weather_tool_schema = {
	"type": "function",
	"name": "get_weather",
	"description": "查询某城市的当前天气",
	"parameters": {
		"type": "object",
		"properties": { 
			"city": {
				"type": "string",
				"description": "城市名"
			}
		},
		"required": ["city"]
	}
}

tools.append(weather_tool_schema)

def get_weather(city : str)->str:
	geo = requests.get(
		"https://geocoding-api.open-meteo.com/v1/search",
		params={"name": city, "count": 1, "language": "zh"},
		timeout=10,
	).json()

	if not geo.get("results"):
		return f"查不到城市「{city}」，换个写法试试"
	loc = geo["results"][0]

	w = requests.get(
		"https://api.open-meteo.com/v1/forecast",
		params={"latitude": loc["latitude"], "longitude": loc["longitude"],
				"current": "temperature_2m,weather_code,wind_speed_10m"},
		timeout=10,
	).json()
	cur = w["current"]
	desc = WEATHER_TEXT.get(cur["weather_code"], f"天气码 {cur['weather_code']}")
	return f"{city} 当前{desc},{cur['temperature_2m']}°C,风速 {cur['wind_speed_10m']} km/h"


def run_agent(task : str,max_step : int = 5)->str:
	input : list = []
	input.append(
		{
			"role" : "user",
			"content" : [
				{
					"type" : "input_text",
					"text" : task,
				},
			]
		}
	)
	resp : Response = None
	for _ in range(max_step):
		resp = client.responses.create(
			input = input,
			instructions = "这是一次循环调用工具的测试,一次回答仅调用一次工具",
			model = "deepseek-flash",
			tools = tools,
		)

		input += resp.output
		tool_calls = [i for i in resp.output if i.type == "function_call"]

		if not tool_calls:
			return f"经过{_}次查询最终答案:" + resp.output_text

		for tool_call in tool_calls:
			args = json.loads(tool_call.arguments)
			result = get_weather(**args)

			input.append(
				{
					"type" : "function_call_output",
					"call_id" : tool_call.call_id,
					"output" : result,
 				}
			)
			print(f"第{_}次,查询结果: {result}")

	return "模型没有找到答案"

task : str = "帮我尝试查找现在中国有没有温度超过40摄氏度的城市"

print(run_agent(task))

### 增加工具数量
增加查询时间，汇率的工具，一轮对话多次调用最后给出综合性的答案

In [ ]:
import os
from openai import OpenAI
from openai.types.responses import Response
from jsonschema import Validator,ValidationError
from pydantic import BaseModel
import requests
import json
import time
from typing import Callable

client = OpenAI(
	api_key = os.environ.get("DEEPSEEK_API_KEY"),
	base_url= "https://api.deepseek.com"
)

WEATHER_TEXT = {
    0: "晴", 1: "基本晴", 2: "多云", 3: "阴", 45: "雾", 48: "雾凇",
    51: "毛毛雨", 53: "毛毛雨", 55: "毛毛雨", 61: "小雨", 63: "中雨", 65: "大雨",
    71: "小雪", 73: "中雪", 75: "大雪", 80: "阵雨", 81: "阵雨", 82: "强阵雨",
    95: "雷暴", 96: "雷暴伴冰雹", 99: "雷暴伴冰雹",
}

tools : list = []
tool_func : dict[str,Callable] = {}

def create_schema(name : str,description : str,properties : dict,required : list[str])->dict:
	return {
		"type": "function",
		"name": name,
		"description": description,
		"parameters": {
			"type": "object",
			"properties": properties,
			"required": required,
		}
	}

tools.append(create_schema(
	name = "get_weather",
	description = "查询某城市的当前天气",
	properties = {
		"city": {
			"type" : "string",
			"description" : "城市名",
		},
	},
	required = ["city"],
))

tools.append(create_schema(
	name = "get_cur_time",
	description = "查询当前时间",
	properties = {},
	required = [],
))

tools.append(create_schema(
	name = "get_exchange_rate",
	description = "查询两种货币汇率",
	properties = {
		"base" : {
			"type" : "string",
			"description" : "货币1",
		},
		"quote" : {
			"type" : "string",
			"description" : "货币2",
		}
	},
	required = ["base","quote"],
))


def get_exchange_rate(base, quote)->str:  return f"1 {base} = 7.2 {quote}"
tool_func["get_exchange_rate"] = get_exchange_rate

def get_cur_time()->str: return f"{time.strftime('%X')}" 
tool_func["get_cur_time"] = get_cur_time

def get_weather(city : str)->str:
	geo = requests.get(
		"https://geocoding-api.open-meteo.com/v1/search",
		params={"name": city, "count": 1, "language": "zh"},
		timeout=10,
	).json()

	if not geo.get("results"):
		return f"查不到城市「{city}」，换个写法试试"
	loc = geo["results"][0]

	w = requests.get(
		"https://api.open-meteo.com/v1/forecast",
		params={"latitude": loc["latitude"], "longitude": loc["longitude"],
				"current": "temperature_2m,weather_code,wind_speed_10m"},
		timeout=10,
	).json()
	cur = w["current"]
	desc = WEATHER_TEXT.get(cur["weather_code"], f"天气码 {cur['weather_code']}")
	return f"{city} 当前{desc},{cur['temperature_2m']}°C,风速 {cur['wind_speed_10m']} km/h"
tool_func["get_weather"] = get_weather


def run_agent(task : str,max_step : int = 5)->str:
	input : list = []
	input.append(
		{
			"role" : "user",
			"content" : [
				{
					"type" : "input_text",
					"text" : task,
				},
			]
		}
	)
	resp : Response = None
	for step in range(max_step):
		resp = client.responses.create(
			input = input,
			instructions = "这是多工具调用的测试",
			model = "deepseek-flash",
			tools = tools,
		)

		input += resp.output
		tool_calls = [i for i in resp.output if i.type == "function_call"]

		if not tool_calls:
			return "最终答案:" + resp.output_text

		for tool_call in tool_calls:
			print(f"第{step}步,调用{tool_call.name} : {tool_call.arguments}")

			try:
				args = json.loads(tool_call.arguments)
				result = tool_func[tool_call.name](**args)
			except Exception as e:
				result = f"调用失败:{e}"

			input.append(
				{
					"type" : "function_call_output",
					"call_id" : tool_call.call_id,
					"output" : result,
 				}
			)
			print(f"第{step}步,查询结果: {result}")

	return "模型没有找到答案"

task : str = "帮我查询北京的天气,现在的时间和人民币和美元的汇率"

print(run_agent(task,2))